# Random Forest Classifier — Notes

Since you've already covered **Decision Trees → Ensemble Learning → Bagging**, Random Forest is the natural next step.

## 1. What is Random Forest?

**Random Forest is an ensemble learning algorithm that combines multiple Decision Trees.**

Instead of using one Decision Tree:

```text
Dataset
   ↓
Decision Tree
   ↓
Prediction
```

Random Forest creates many trees:

```text
                 Dataset
                    │
        ┌───────────┼───────────┐
        ↓           ↓           ↓
      Tree 1      Tree 2      Tree 3   ... Tree N
        ↓           ↓           ↓
       Cat         Dog         Cat
        └───────────┼───────────┘
                    ↓
               Majority Vote
                    ↓
                Final Class
```

For classification:

> **Final prediction = majority vote of all trees**

---

# 2. Why Random Forest?

A single Decision Tree can easily **overfit**.

For example:

```text
Decision Tree
Training accuracy = 100%
Testing accuracy  = 82%
```

Random Forest reduces this problem by combining many different trees.

The main idea is:

> **Many diverse trees + aggregation = better generalization**

---

# 3. Random Forest = Bagging + Random Feature Selection

This is the most important concept.

Random Forest introduces **two types of randomness**.

### Randomness 1 — Random samples

Each tree receives a different bootstrap sample of the training data.

```text
Original Dataset
      │
      ├── Bootstrap sample → Tree 1
      ├── Bootstrap sample → Tree 2
      ├── Bootstrap sample → Tree 3
      └── Bootstrap sample → Tree N
```

This is **Bagging**.

---

### Randomness 2 — Random features

When a tree is deciding which feature to split on, Random Forest considers only a **random subset of features**.

Suppose you have:

```text
10 features
```

A particular split might only consider:

```text
Feature 2
Feature 4
Feature 7
Feature 9
```

rather than all 10.

This makes trees more **diverse/decorrelated**.

---

# 4. Why Random Feature Selection?

Imagine your dataset has:

```text
X1 = very strong feature
X2
X3
X4
X5
...
```

If every tree always sees `X1`, many trees can become very similar.

Random Forest forces trees to explore different features.

Therefore:

```text
Tree 1 → different
Tree 2 → different
Tree 3 → different
Tree 4 → different
```

This reduces correlation between trees.

And lower correlation between strong learners generally improves ensemble performance.

---

# 5. Bootstrap Sampling

Suppose your training dataset contains:

```text
10 samples
```

Random Forest randomly samples from those 10 **with replacement**.

Example:

```text
Original:

A B C D E F G H I J

Tree 1:

A C C D F H H I J J

Tree 2:

B B D E F G G I I J

Tree 3:

A A C E E F H I J J
```

Notice:

* Samples can appear multiple times.
* Some samples aren't selected.
* Every tree gets a different dataset.

This is called **bootstrap sampling**.

---

# 6. What happens to samples not selected?

These are called **Out-of-Bag (OOB) samples**.

For each tree:

```text
Bootstrap samples → used for training
OOB samples       → not used for training
```

Random Forest can use these OOB samples to estimate model performance.

This gives us:

> **OOB Score**

---

# 7. `oob_score`

Example:

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    oob_score=True,
    random_state=42
)

rf.fit(X_train, y_train)

print(rf.oob_score_)
```

Example:

```text
0.94
```

Approximately:

```text
OOB accuracy = 94%
```

### Important

OOB evaluation can be useful because you don't need a separate validation set solely for this purpose.

But you should still keep a proper **test set** for final evaluation.

---

# 8. Random Forest Classification

Scikit-learn:

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
```

Evaluate:

```python
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(accuracy)
```

---

# 9. Important Random Forest Parameters

These are the parameters you should know.

## `n_estimators`

Number of trees.

```python
RandomForestClassifier(
    n_estimators=100
)
```

Means:

```text
100 Decision Trees
```

More trees generally make the prediction more stable, but increase computation time.

Common values:

```text
50
100
200
500
```

---

## `criterion`

Controls how split quality is measured.

Common options:

```python
criterion="gini"
```

or:

```python
criterion="entropy"
```

or in newer sklearn versions:

```python
criterion="log_loss"
```

Example:

```python
rf = RandomForestClassifier(
    criterion="entropy"
)
```

---

# 10. `max_depth`

Controls maximum depth of each tree.

```python
max_depth=5
```

means:

```text
Tree can grow at most 5 levels deep
```

### Small depth

```text
max_depth = 3
```

→ simpler trees
→ potentially underfitting

### Large depth

```text
max_depth = None
```

→ trees can grow very deep
→ potentially overfitting

Random Forest can tolerate fairly deep trees because of ensemble averaging, but controlling depth can still improve speed/generalization.

---

# 11. `min_samples_split`

Minimum number of samples required to split an internal node.

```python
min_samples_split=2
```

Default behavior allows splitting with very few samples.

Increasing it:

```python
min_samples_split=10
```

makes trees more conservative.

---

# 12. `min_samples_leaf`

Minimum number of samples allowed in a leaf.

```python
min_samples_leaf=1
```

versus:

```python
min_samples_leaf=5
```

With:

```python
min_samples_leaf=5
```

every leaf must contain at least 5 samples.

This can reduce overfitting.

---

# 13. `max_features`

**Very important for Random Forest.**

It determines how many features are randomly considered when searching for a split.

For example:

```python
max_features="sqrt"
```

For classification, this is a common choice.

If you have:

```text
100 features
```

then:

```text
sqrt(100) = 10
```

Approximately 10 features are considered at a split.

Other options include:

```python
max_features="sqrt"
max_features="log2"
max_features=None
```

or a number:

```python
max_features=10
```

---

# 14. `bootstrap`

Controls whether bootstrap samples are used.

```python
bootstrap=True
```

This is the classic Random Forest approach.

```python
bootstrap=False
```

means each tree uses the whole training dataset rather than bootstrap sampling.

---

# 15. `class_weight`

Useful for **imbalanced classification datasets**.

Example:

```python
RandomForestClassifier(
    class_weight="balanced"
)
```

Suppose:

```text
Class 0 → 9500 samples
Class 1 → 500 samples
```

The model may otherwise favor Class 0.

`balanced` automatically adjusts class weights based on class frequencies.

---

# 16. `random_state`

Used for reproducibility.

```python
random_state=42
```

Without it, results can vary between runs because Random Forest contains randomness.

---

# 17. Complete Basic Example

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
```

---

# 18. Does Random Forest Need Scaling?

Usually:

> **No.**

Decision Trees are based on conditions such as:

```text
Age <= 35
Income <= 50000
```

They don't care about the magnitude of features in the same way KNN, SVM, or Logistic Regression do.

Therefore:

```text
StandardScaler
MinMaxScaler
```

are generally **not required** for Random Forest.

For example:

```python
X_train
```

can be used directly:

```python
rf.fit(X_train, y_train)
```

---

# 19. Feature Importance

One major advantage of Random Forest is feature importance.

```python
rf.feature_importances_
```

Example:

```python
for feature, importance in zip(
    X.columns,
    rf.feature_importances_
):
    print(feature, importance)
```

Output:

```text
age       0.12
income    0.25
balance   0.31
tenure    0.08
...
```

Higher value:

> Feature contributed more to the model's splitting decisions.

However, impurity-based feature importance can be biased, especially toward high-cardinality features. For serious interpretation, consider **permutation importance** or SHAP.

---

# 20. Random Forest vs Decision Tree

| Decision Tree      | Random Forest                  |
| ------------------ | ------------------------------ |
| One tree           | Many trees                     |
| High variance      | Lower variance                 |
| Can overfit easily | Usually more robust            |
| Fast training      | More computationally expensive |
| Easy to visualize  | Difficult to visualize         |
| Less stable        | More stable                    |
| No ensemble        | Ensemble                       |
| One prediction     | Aggregated prediction          |

The key idea:

```text
Decision Tree
     ↓
High variance

Random Forest
     ↓
Many trees
     ↓
Average / majority vote
     ↓
Reduced variance
```

---

# 21. Hyperparameter Tuning

Now we get to the part you specifically asked about.

Instead of manually deciding:

```python
n_estimators=100
max_depth=10
min_samples_split=2
```

we can let Scikit-learn search for a good combination.

Two important methods:

```text
GridSearchCV
RandomizedSearchCV
```

---

# 22. GridSearchCV

Suppose we want to test:

```python
n_estimators = [100, 200, 300]

max_depth = [None, 10, 20]

min_samples_split = [2, 5]
```

Grid Search tests **every combination**.

Number of combinations:

```text
3 × 3 × 2 = 18
```

Then with:

```python
cv=5
```

each combination is evaluated using 5-fold CV.

Therefore:

```text
18 combinations × 5 folds
= 90 model fits
```

---

# 23. GridSearchCV Example

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(
    random_state=42
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)
```

Then:

```python
print(grid_search.best_params_)
```

Example:

```text
{
    'max_depth': 20,
    'max_features': 'sqrt',
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 200
}
```

Best CV score:

```python
print(grid_search.best_score_)
```

Best model:

```python
best_rf = grid_search.best_estimator_
```

Then evaluate on your untouched test set:

```python
y_pred = best_rf.predict(X_test)

print(accuracy_score(y_test, y_pred))
```

---

# 24. VERY IMPORTANT: `best_score_` vs Test Accuracy

Don't confuse these.

```python
grid_search.best_score_
```

= best **cross-validation score on training data**

while:

```python
accuracy_score(y_test, y_pred)
```

= performance on your **unseen test data**.

Correct workflow:

```text
Dataset
   ↓
Train/Test Split
   ↓
Training Data
   ↓
GridSearchCV
   ↓
Best Hyperparameters
   ↓
Best Model
   ↓
Test Data
   ↓
Final Evaluation
```

Don't use the test set during hyperparameter tuning.

---

# 25. Why GridSearchCV Becomes Slow

This is especially important for Random Forest.

Suppose:

```python
param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}
```

Number of combinations:

```text
4 × 4 × 3 × 3 = 144
```

With:

```python
cv=5
```

you get:

```text
144 × 5 = 720 fits
```

And each fit itself may train hundreds of trees.

That's why it can become **very slow**.

---

# 26. `n_jobs=-1`

For Random Forest and GridSearchCV, you will frequently see:

```python
n_jobs=-1
```

This means:

> Use all available CPU cores where supported.

Example:

```python
GridSearchCV(
    rf,
    param_grid,
    cv=5,
    n_jobs=-1
)
```

And:

```python
RandomForestClassifier(
    n_estimators=300,
    n_jobs=-1
)
```

can parallelize tree building.

---

# 27. What About Bigger Datasets?

This is where you should **not blindly use GridSearchCV**.

Imagine:

```text
1,000,000 rows
100 features
```

and you try:

```text
GridSearchCV
    ↓
100 combinations
    ↓
5-fold CV
    ↓
500 fits
    ↓
each fit = 500 trees
```

This can become extremely expensive.

Instead, use a more efficient search strategy.

---

# 28. RandomizedSearchCV

Instead of testing **every combination**, RandomizedSearchCV randomly samples a fixed number of combinations.

Example:

```python
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=2
)

random_search.fit(X_train, y_train)
```

Here:

```python
n_iter=20
```

means:

> Test only 20 randomly selected parameter combinations.

So instead of:

```text
100 combinations × 5
= 500 fits
```

you get:

```text
20 × 5
= 100 fits
```

That's a **5× reduction** in the number of CV fits.

---

# 29. GridSearch vs RandomizedSearch

| GridSearchCV                   | RandomizedSearchCV             |
| ------------------------------ | ------------------------------ |
| Tests every combination        | Tests random combinations      |
| Can be expensive               | Usually faster                 |
| Good for small search space    | Good for large search space    |
| Exhaustive                     | Non-exhaustive                 |
| More computationally expensive | More computationally efficient |
| `param_grid`                   | `param_distributions`          |
| No `n_iter`                    | Uses `n_iter`                  |

---

# 30. What is `n_iter`?

This connects directly to your earlier question.

```python
n_iter=20
```

means:

> RandomizedSearchCV will evaluate **20 parameter configurations**.

It does **not** mean 20 individual models if you use CV.

For:

```python
n_iter=20
cv=5
```

you get approximately:

```text
20 × 5
= 100 model fits
```

---

# 31. Better Strategy for Large Datasets

For a large dataset, I would generally use:

```text
                    Large Dataset
                         │
                         ↓
                Train / Validation / Test
                         │
                         ↓
                RandomizedSearchCV
                         │
                         ↓
                Narrow parameter range
                         │
                         ↓
                Best parameters
                         │
                         ↓
                  Optional GridSearch
                         │
                         ↓
                  Final model
                         │
                         ↓
                    Test evaluation
```

You don't need to perform a giant GridSearch from the beginning.

---

# 32. Practical Large-Dataset Search

For example:

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=2
)

random_search.fit(X_train, y_train)
```

Then:

```python
print(random_search.best_params_)
print(random_search.best_score_)
```

And:

```python
best_rf = random_search.best_estimator_

y_pred = best_rf.predict(X_test)
```

---

# 33. Even Faster Approach for Huge Datasets

If the dataset is **really large**, even RandomizedSearchCV with `cv=5` can be expensive.

You can reduce:

### Number of iterations

```python
n_iter=10
```

instead of:

```python
n_iter=100
```

### CV folds

```python
cv=3
```

instead of:

```python
cv=5
```

### Number of trees during tuning

Use:

```python
n_estimators=[100, 200]
```

during the search.

After finding good hyperparameters, train the final model with more trees:

```python
RandomForestClassifier(
    n_estimators=500,
    ...
)
```

This is often much more practical.

---

# 34. Successive Halving

For very large hyperparameter searches, another approach is **successive halving**.

Idea:

```text
Many parameter combinations
          ↓
Small amount of resources
          ↓
Remove poor configurations
          ↓
More resources for survivors
          ↓
Remove more
          ↓
Best configurations
```

Scikit-learn provides:

```python
HalvingRandomSearchCV
```

and:

```python
HalvingGridSearchCV
```

These can be useful when the search space is large and individual model training is expensive.

---

# 35. Important Point About Random Forest and Large Data

Random Forest itself is highly parallelizable.

This:

```python
RandomForestClassifier(
    n_estimators=500,
    n_jobs=-1
)
```

can train trees in parallel.

But don't confuse:

```text
n_jobs
```

with:

```text
n_estimators
```

### `n_estimators`

Number of trees.

### `n_jobs`

Number of CPU workers used for parallel computation.

---

# 36. Recommended Workflow for You

Since you're learning ensemble learning, I'd follow this progression:

```text
Decision Tree
      ↓
Bagging
      ↓
Random Forest
      ↓
Random Forest parameters
      ↓
GridSearchCV
      ↓
RandomizedSearchCV
      ↓
Large dataset optimization
      ↓
Extra Trees
      ↓
Boosting
```

And for Random Forest specifically, master these first:

```text
n_estimators
max_depth
max_features
min_samples_split
min_samples_leaf
bootstrap
class_weight
oob_score
n_jobs
random_state
```

### The key mental model

```text
Random Forest
=
Bagging
+
Random feature selection
+
Multiple Decision Trees
+
Aggregation
```

And for tuning:

```text
Small dataset
    → GridSearchCV

Large search space / larger dataset
    → RandomizedSearchCV

Very large/expensive search
    → Successive Halving / more efficient tuning
```

One especially important lesson from your recent GridSearchCV work: **the number of hyperparameter combinations × CV folds is only the number of fits; for Random Forest, each fit may itself contain hundreds of trees.** That's why a seemingly modest search can explode computationally.
